In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd

import statsmodels.api as sm
import statsmodels.formula.api as smf

from ISLP.models import (ModelSpec as MS, summarize , poly)

# needed for kNN
from sklearn.neighbors import KNeighborsRegressor

# Linear Regression with Nonlinear Features
Given data with a nonlinear relationship between $X$ and $Y$, try to learn the covariates for
$$
Y = \beta_0 + \beta_1 X + \beta_2 X^2 + \beta_3 X^3 + \epsilon
$$
This assumes that we have good reason to pick degree three polynomials.  

## Construct Data

In [ ]:
n = 50
rng = np.random.default_rng(1234)

x_ = rng.uniform(-2, 2, n)
y_ = 3 - 3* x_ + x_**3 +  0.5 * rng.normal(size=x_.shape)

cubic_df = pd.DataFrame({'x': x_, 'y': y_})
cubic_df.head()
fig, ax = plt.subplots()
cubic_df.plot.scatter('x', 'y', ax=ax, label='Data')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$')
ax.legend()
# fig.savefig('cubic_data.pdf')

## Set Up Model and Train

In [ ]:
design = MS([ poly('x', degree =3)])
X = design.fit_transform(cubic_df)
X.head()


**NOTE** These features do _not_ correspond to the $(1, x_i, x_i^2, x_i^3)$

In [ ]:
x_[0:4]

This is done to stabilize the numerical linear algebra that shows up to solve the normal equations:
$$
\mathcal{X}^T\mathcal{X} \hat{\beta} = \mathcal{X}^Ty
$$
For pedagogical reasons, we can access the less stable, but more explicit repreesntations with the `raw=True` flag:

In [ ]:
design = MS([ poly('x', degree =3, raw=True)])
X = design.fit_transform(cubic_df)
X.head()


In [ ]:
print(cubic_df['x'].head())
print(cubic_df['x'].head()**2)
print(cubic_df['x'].head()**3)

These now agree with the pencil/paper representation, even though we would not use them in real problems.

In [ ]:
model = sm.OLS(cubic_df['y'], X)
results = model.fit()
summarize(results)

These numbers are consistent with our true polynomial, $3 - 3x +x^3$.  Observe that with the $t$-statistic for the degree two term, we would accept the null hypothesis that this was really zero.

## Visualize

In [ ]:
xx = np.linspace(-2,2,100)
new_df = pd.DataFrame({'x': xx})
newX = design.transform(new_df)
newY = results.predict(newX)
fig, ax = plt.subplots()
cubic_df.plot.scatter('x', 'y', ax=ax, label='Data')
ax.plot(xx, newY, color='C1', label='OLS Fit')
ax.plot(xx, 3 - 3* xx + xx**3, color='black', linestyle='--', label='Truth')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$')
ax.legend()


Let us plot the 95% CIs and PIs along with this:

In [ ]:

# get predictions at evaluation points
preds = results.get_prediction(newX) 

# get the confidence intervals and prediction intervals
ci_vals = preds.conf_int()
pi_vals = preds.conf_int(obs=True)

fig, ax = plt.subplots()
cubic_df.plot.scatter('x', 'y', ax=ax, label='Data')
ax.plot(xx, newY, color='C1', label='OLS Fit')
ax.fill_between(
    new_df['x'],
    ci_vals[:,0], # lower bound
    ci_vals[:,1], # upper bound
    color='C1',
    alpha=0.25,     # transparency
    label='95% CI',
)
ax.fill_between(
    new_df['x'],
    pi_vals[:,0], # lower bound
    pi_vals[:,1], # upper bound
    color='C2',
    alpha=0.25,     # transparency
    label='95% PI',
)
ax.plot(xx, 3 - 3* xx + xx**3, color='black', linestyle='--', label='Truth')

ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$')
ax.legend()



The confidence interval does a fairly good job of capturing the truth, and the prediction interval does a good job of capturing the data.

# kNN Regression
Given training data and a choice of $K$, we will make new predictions via
$$
\hat{f}(x) = \frac{1}{K}\sum_{x_i \in \mathcal{N}_x} y_i,
$$
where $\mathcal{N}_x$ contains the $K$ $x_i$'s closets (in Euclidean distance) to $x$

## Construct Data
For this to work, we will need more data:

In [ ]:
n = 500
rng = np.random.default_rng(1234)

x_ = rng.uniform(-2, 2, n)
# y_ = 3 - 3* x_ + x_**3 +  0.4 * rng.normal(size=x_.shape)
y_ = 3 - 3* x_ + x_**3 +  0.5 * rng.normal(size=x_.shape)

cubic_df = pd.DataFrame({'x': x_, 'y': y_})

## Set up Model
There is no training (fitting); we just specify the training data and the number of neighbors.  Experiment with the value of $K$

In [ ]:
knn = KNeighborsRegressor(n_neighbors=10) # K = 10 here
knn.fit(cubic_df[['x']], cubic_df['y'])

## Visualize

In [ ]:
xx = np.linspace(-2,2,100)
new_df = pd.DataFrame({'x': xx})
yhat = knn.predict(new_df[['x']])

In [ ]:
fig, ax = plt.subplots()
cubic_df.plot.scatter('x', 'y', ax=ax, label='Data')
ax.plot(new_df['x'], yhat, color='C1', label='KNN Fit')
ax.legend()